# Communicatie & Contactgegevens

**Thema:** Contactinformatie van personen en commissies.

**Kernvragen:** Welke soorten contactkanalen (e-mail, telefoon, sociale media, websites) worden het meest gebruikt door commissies en personen, en hoe is dit gestructureerd op basis van gewichten?

Dit notebook verkent o.a. de silver-tabellen `persoon`, `persoon_contactinformatie`, `commissie` en `commissie_contactinformatie`.

In [ ]:
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

def find_database(start: Path | None = None) -> Path:
    """Find the repository DuckDB database by walking up from the current folder."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        candidate = folder / "tweedekamer.duckdb"
        if candidate.is_file():
            return candidate
    raise FileNotFoundError("Could not find tweedekamer.duckdb from the current directory")


database_path = find_database()
connection = duckdb.connect(str(database_path), read_only=True)


def query_df(sql: str, parameters=None) -> pd.DataFrame:
    """Execute a SQL query and return the result as a pandas DataFrame."""
    return connection.execute(sql, parameters or []).fetchdf()


database_path

## Onderwerp 1

Welk type contact wordt het vaakst gebruikt per persoon?

Om er achter te komen welk soort contactinformatie het meest wordt gebruikt door kamerleden, wil je graag het gemiddelde weten van deze de soorten. Dit is bepaald doormiddel van de volgende code:

In [ ]:
contacten = query_df("""
  SELECT DISTINCT soort, AVG(gewicht) AS gemiddeld_gewicht
  FROM silver.persoon_contactinformatie AS A
  GROUP BY soort
  ORDER BY gemiddeld_gewicht DESC
""")
contacten

contacten.plot(
    x="soort",
    y="gemiddeld_gewicht",
    kind="bar",
    legend=False,
    figsize=(10, 6)
)

plt.title("Gemiddeld gewicht per soort contactinformatie")
plt.xlabel("Soort")
plt.ylabel("Gemiddeld gewicht")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Zoals is te zien wordt gemiddeld gezien Bluesky het meest gebruikt en E-mail gemiddeld het minst. Hieruit kan je echter nog niet concluderen welk soort contactinformatie het meest wordt gebruikt. Een kamerlid kan bijvoorbeeld welk erg vaak gebruikt maken van Bluesky, terwijl de rest van de kamer helemaal niet gebruik maken van Bluesky. Daarom is het ook verstandig om te kijken naar het aantal kamerleden dat gebruikt maakt van bepaalde soorten contactinformaties

In [ ]:
contacten_2 = query_df("""
  SELECT soort, COUNT(soort) AS aantal
  FROM silver.persoon_contactinformatie AS A
  GROUP BY soort
  ORDER BY aantal DESC
""")
contacten_2

contacten_2.plot(
    x="soort",
    y="aantal",
    kind="bar",
    legend=False,
    figsize=(10, 6)
)

plt.title("Aantal soort contactinformatie")
plt.xlabel("Soort")
plt.ylabel("Aantal")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In contrast met het gemiddelde gewicht van contactinformaties, maken meer kamerleden gebruik van Email vergeleken met Bluesky. Daarnaast is het verschil tussen E-mail en de nummer twee, Website, aardig groot. Dit is te verwachten omdat E-mail erg belangrijk is in het dagelijks leven, terwijl bijvoorbeeld X of Bluesky dat niet zijn.

## Onderwerp 2

Welke soort contactinformaties worden er gebruikt in commissies en welke wordt het meest gebruikt?

Vervolg op onderwerp 2, alleen nu wordt er gekeken naar het gebruik van soorten contactkanalen door commissies. Al snel wordt er gezien dat er meer soorten contactgegevens zijn.

In [ ]:
soorten = query_df("""
    SELECT DISTINCT soort
    FROM silver.commissie_contactinformatie
""")

soorten

In [ ]:
contacten = query_df("""
  SELECT DISTINCT soort, AVG(gewicht) AS gemiddeld_gewicht
  FROM silver.commissie_contactinformatie AS A
  GROUP BY soort
  ORDER BY gemiddeld_gewicht DESC
""")
contacten

contacten.plot(
    x="soort",
    y="gemiddeld_gewicht",
    kind="bar",
    legend=False,
    figsize=(10, 6)
)

plt.title("Gemiddeld gewicht per soort contactinformatie")
plt.xlabel("Soort")
plt.ylabel("Gemiddeld gewicht")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Deze grafiek geeft aan dat twitter gemiddeld gezien het meest wordt gebruikt terwijl er het minst gebruikt wordt gemaakt van een Griffier. Wel is op te merken dat de taak van een adjunct-griffier vaker voorkomd. De baan van een adjunct-griffier is om een griffier te ondersteunen. En bij afwezigheid van een griffier, de taak over te nemen. 

Ook bij dit gedeelte wordt er ook gekeken wat het aantal gebruik is van soorten contactkanalen. Resultaten zijn als volgt:

In [ ]:
contacten_2 = query_df("""
  SELECT soort, COUNT(soort) AS aantal
  FROM silver.commissie_contactinformatie AS A
  GROUP BY soort
  ORDER BY aantal DESC
""")
contacten_2

contacten_2.plot(
    x="soort",
    y="aantal",
    kind="bar",
    legend=False,
    figsize=(10, 6)
)

plt.title("Aantal soort contactinformatie")
plt.xlabel("Soort")
plt.ylabel("Aantal")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Onderwerp 3

Welke personen maakt het meest gebruik van internet?

In [ ]:
meestgebruik = query_df("""
    SELECT voornamen, achternaam, fractielabel, SUM(gewicht) AS hoeveelheid_gebruik
    FROM silver.persoon_contactinformatie AS A
    JOIN silver.persoon AS B
    ON B.id = A.persoon_id
    GROUP BY voornamen, achternaam, fractielabel
    ORDER BY hoeveelheid_gebruik DESC
    LIMIT 20
""")

meestgebruik

In [ ]:
meestgebruik.plot(
    x="achternaam",
    y="hoeveelheid_gebruik",
    kind="bar",
    legend=False,
    figsize=(10, 6)
)

plt.title("Meeste gebruik van internet")
plt.xlabel("Personen")
plt.ylabel("Gewicht")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

Jammer genoeg hebben bovenstaande mensen niet aangegeven in fractielabel bij welke partijen ze zitten. 

## Onderwerp 4

Welke contact soort gebruikt elke partij het meest? 

In [ ]:
meest_gebruikt = query_df("""
    WITH contact_per_partij AS (
        SELECT
            B.fractielabel AS partij,
            A.soort AS contactsoort,
            SUM(A.gewicht) AS totaal_gewicht
        FROM silver.persoon_contactinformatie AS A
        INNER JOIN silver.persoon AS B
            ON A.persoon_id = B.id
        WHERE B.fractielabel IS NOT NULL
          AND LOWER(TRIM(B.fractielabel)) <> 'nan'
          AND A.soort IS NOT NULL
        GROUP BY
            B.fractielabel,
            A.soort
    ),
    rangschikking AS (
        SELECT
            partij,
            contactsoort,
            totaal_gewicht,
            ROW_NUMBER() OVER (
                PARTITION BY partij
                ORDER BY totaal_gewicht DESC
            ) AS rang
        FROM contact_per_partij
    )
    SELECT
        partij,
        contactsoort,
        totaal_gewicht
    FROM rangschikking
    WHERE rang = 1
    ORDER BY partij
""")

meest_gebruikt

Na deze tabel moet nog een keer gekeken worden, maar aannemend dat de tabel klopt springt de VVD eruit met Twitter. 

## Ontwerp 5

Is er een verschil in het gebruik van contact kanalen tussen mannen en vrouwen?

In [ ]:
geslacht = query_df("""
    SELECT geslacht, AVG(gewicht) AS gemiddeld_gewicht
    FROM silver.persoon AS A
    JOIN silver.persoon_contactinformatie AS B
    ON A.id = B.persoon_id
    GROUP BY geslacht
""")
geslacht

In [ ]:
geslacht.sort_values(
    "gemiddeld_gewicht",
    ascending=True
).plot(
    x="geslacht",
    y="gemiddeld_gewicht",
    kind="barh",
    legend=False,
    figsize=(10, 7)
)

plt.title("Gemiddeld gebruik per geslacht")
plt.xlabel("Gemiddeld gewicht")
plt.ylabel("Geslacht")
plt.tight_layout()
plt.show()

Er is hier niet heel veel op aan te merken. De waardes liggen erg dicht bij elkaar.

## Ontwerp 6

Welke commissie maakt het meest gebruik van contactkanalen?

In [ ]:
top10_commissie = query_df("""
    SELECT naam_nl, SUM(gewicht) AS totaal_gebruik
    FROM silver.commissie AS A
    JOIN silver.commissie_contactinformatie AS B
    ON A.id = B.commissie_id
    GROUP BY naam_nl
    ORDER BY totaal_gebruik DESC
    LIMIT 10
""")
top10_commissie

In [ ]:
top10_commissie.sort_values(
    "totaal_gebruik",
    ascending=True
).plot(
    x="naam_nl",
    y="totaal_gebruik",
    kind="barh",
    legend=False,
    figsize=(10, 7)
)

plt.title("Meest gebruik gemaakt van soorten contactkanalen door commissies")
plt.xlabel("Totaal gebruik")
plt.ylabel("Commissie naam")
plt.tight_layout()
plt.show()

Uit deze grafiek kan geconcludeerd worden dat de commissie van Buitenlandse handel en ontwikkelingssamenwerking het meest gebruikt maakt van soorten contact kanalen. Aangezien de wereldhandel snel veranderd is dit wel redelijk logisch.

## Interesante opmerkingen

Niet alle bevindingen zijn bruikbaar en zijn daarom ook niet handig om mee te nemen in de ontwerpen. Daarentegen geven deze bevindingen wel een beter beeld van de data. Daarnaast is het ook wel interesant.

In [ ]:
vergaderzalen = query_df("""
  SELECT voornamen, achternaam, soort, gewicht, waarde
  FROM silver.persoon_contactinformatie AS A
  JOIN silver.persoon AS B
  ON A.persoon_id = B.id
  WHERE voornamen = 'Geert'
  ORDER BY gewicht DESC
  LIMIT 20
""")
 
vergaderzalen

In [ ]:
query_df("""
    SELECT MAX(gewicht)
    FROM silver.persoon_contactinformatie
""")

Deze codes gaven mij een beter inzicht van de data. Ik wist niet in het begin wat er met gewicht werdt bedoeld, maar naar deze bevindingen te kijken denk ik wel dat ik een beter beeld heb gekregen. Het gewicht is waarschijnlijk een persoonlijke ranglijst in hoeveel een bepaalde soort contactkanaal wordt gebruikt. Bij Geert Wilders is te zien dat hij vind dat hij erg veel Instagram gebruikt en maar heel weinig E-mail. Aangezien de maximale waarde 7 is, blijkt hieruit dat Geert Wilders erg neutral is over zijn Twitter gebruik terwijl hij juist erg vaak Twittert.

In [ ]:
query_df("""
    SELECT *
    FROM silver.persoon_contactinformatie
    LIMIT 10
""")